# python + sql integration

generate a summary report based on the selected report type and date range.

##Step 1: User Input (Python Cell)

In [0]:
report_type = "monthly"

start_date = "2025-01-01"

end_date = "2026-12-31"

##Step 2: Total Orders

In [0]:
total_orders = spark.sql(f"""

select count(*) as total_orders
from orders_clean
where order_date between '{start_date}' and '{end_date}'

""")

total_orders.show()

+------------+
|total_orders|
+------------+
|         683|
+------------+



##Step 3: Total Revenue

In [0]:
total_revenue = spark.sql(f"""

select
    round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)),2) as total_revenue
from orders_clean o
join order_items_clean oi
    on o.order_id = oi.order_id
join products_clean p
    on oi.product_id = p.product_id
where o.order_date between '{start_date}' and '{end_date}'

""")

total_revenue.show()

+-------------+
|total_revenue|
+-------------+
|7.438300359E7|
+-------------+



##Step 4: Unique Customers

In [0]:
unique_customers = spark.sql(f"""

select
    count(distinct customer_id) as unique_customers
from orders_clean
where order_date between '{start_date}' and '{end_date}'
and customer_id <> -1

""")

unique_customers.show()

+----------------+
|unique_customers|
+----------------+
|             455|
+----------------+



##Step 5: Top 3 Products

In [0]:
top_products = spark.sql(f"""

select
    p.product_name,
    sum(abs(oi.quantity)) as total_quantity
from orders_clean o
join order_items_clean oi
    on o.order_id = oi.order_id
join products_clean p
    on oi.product_id = p.product_id
where o.order_date between '{start_date}' and '{end_date}'
group by p.product_name
order by total_quantity desc
limit 3

""")

top_products.show()

+-------------+--------------+
| product_name|total_quantity|
+-------------+--------------+
| Water Bottle|           234|
|Atomic Habits|           204|
| Sql Cookbook|           190|
+-------------+--------------+



##Step 6: Previous Period

First, calculate how many days are in the selected period.

In [0]:
from datetime import datetime, timedelta

start = datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.strptime(end_date, "%Y-%m-%d")

days = (end - start).days + 1

previous_start = (start - timedelta(days=days)).strftime("%Y-%m-%d")
previous_end = (start - timedelta(days=1)).strftime("%Y-%m-%d")

print(previous_start)
print(previous_end)

2023-01-02
2024-12-31


##Step 7: Current Revenue

In [0]:
current_revenue = spark.sql(f"""

select
    round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)),2) as revenue
from orders_clean o
join order_items_clean oi
    on o.order_id = oi.order_id
join products_clean p
    on oi.product_id = p.product_id
where o.order_date between '{start_date}' and '{end_date}'

""").collect()[0]["revenue"]

##Step 8: Previous Revenue

In [0]:
previous_revenue = spark.sql(f"""

select
    round(sum(oi.quantity * p.unit_price * (1 - oi.discount_percent / 100)),2) as revenue
from orders_clean o
join order_items_clean oi
    on o.order_id = oi.order_id
join products_clean p
    on oi.product_id = p.product_id
where o.order_date between '{previous_start}' and '{previous_end}'

""").collect()[0]["revenue"]

##Step 9: Percentage Change

In [0]:
if previous_revenue and previous_revenue != 0:
    revenue_change = round(
        ((current_revenue - previous_revenue) / previous_revenue) * 100,
        2
    )
else:
    revenue_change = 0

## Step 10: Final Report

In [0]:
print("=" * 40)
print("E-Commerce Summary Report")
print("=" * 40)

print(f"Report Type      : {report_type}")
print(f"Date Range       : {start_date} to {end_date}")
print(f"Revenue          : {current_revenue}")
print(f"Revenue Change   : {revenue_change}%")

E-Commerce Summary Report
Report Type      : monthly
Date Range       : 2025-01-01 to 2026-12-31
Revenue          : 74383003.59
Revenue Change   : 242.7%
